# LIMUC supervised fine-tuning (ResNet-50)
Fine-tune ResNet-50 on LIMUC UC severity (MES) with ordinal-aware metrics.


In [1]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms


In [2]:
# Paths & config
def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
LABEL_MAP_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "label_map.csv"
OUT_DIR = DATA_ROOT / "2_supervised_finetuning" / "out" / "finetune_resnet50"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.getenv("SEED", "42"))
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "16"))
NUM_WORKERS = int(os.getenv("NUM_WORKERS", "0"))
EPOCHS = int(os.getenv("EPOCHS", "15"))
LR = float(os.getenv("LR", "3e-4"))
WEIGHT_DECAY = float(os.getenv("WEIGHT_DECAY", "1e-4"))
MAX_SAMPLES = int(os.getenv("MAX_SAMPLES", "0")) or None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Output dir:", OUT_DIR)


Device: cuda
Output dir: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/LIMUC/2_supervised_finetuning/out/finetune_resnet50


In [3]:
# Seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Load metadata
meta = pd.read_csv(META_CSV)
images_base = DATA_ROOT / "0_dataset_prep"

def to_abs(p):
    p = Path(p)
    if p.is_absolute():
        return p
    return (images_base / p).resolve()

meta["image_path"] = meta["image_path"].apply(lambda p: str(to_abs(p)))

# Label map
if LABEL_MAP_CSV.exists():
    label_map = pd.read_csv(LABEL_MAP_CSV)
    id_to_name = dict(zip(label_map.label_id, label_map.label_name))
else:
    id_to_name = {i: name for i, name in enumerate(sorted(meta.label_name.unique()))}

name_to_id = {v: k for k, v in id_to_name.items()}
meta["label_id"] = meta["label_name"].map(name_to_id)

meta = meta[meta["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
if MAX_SAMPLES:
    meta = meta.sample(n=min(MAX_SAMPLES, len(meta)), random_state=SEED).reset_index(drop=True)

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "val"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))


Train/Val/Test: 8669 921 1686


In [4]:
# Transforms (mild augmentations)
train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class ImageDS(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.paths = df["image_path"].tolist()
        self.labels = df["label_id"].tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        label = int(self.labels[idx])
        return img, label


In [5]:
# Dataloaders
train_dl = DataLoader(ImageDS(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_dl = DataLoader(ImageDS(val_df, val_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_dl = DataLoader(ImageDS(test_df, val_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Class weights
class_counts = train_df["label_id"].value_counts().sort_index()
num_classes = len(class_counts)
total = class_counts.sum()
weights = total / (num_classes * class_counts)
class_weights = torch.tensor(weights.values, dtype=torch.float).to(DEVICE)

# Model
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


In [6]:
def run_epoch(dataloader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    for x, y in dataloader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(is_train):
            logits = model(x)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        preds = probs.argmax(axis=1)

        total_loss += loss.item() * y.size(0)
        all_probs.append(probs)
        all_preds.extend(preds)
        all_labels.extend(y.detach().cpu().numpy())

    avg_loss = total_loss / max(len(dataloader.dataset), 1)
    all_probs = np.concatenate(all_probs, axis=0) if all_probs else None
    return avg_loss, np.array(all_labels), np.array(all_preds), all_probs


In [7]:
# =====================
# Metrics helpers
# =====================
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)

try:
    from scipy.stats import spearmanr
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False


def expected_calibration_error(y_true, y_prob, n_bins=10):
    if y_prob is None:
        return None
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    confidences = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i + 1])
        if mask.any():
            ece += abs(accuracies[mask].mean() - confidences[mask].mean()) * mask.mean()
    return float(ece)


def compute_metrics(y_true, y_pred, labels, label_names, y_prob=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )

    summary = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "qwk": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

    if _HAS_SCIPY:
        summary["spearman"] = float(spearmanr(y_true, y_pred).correlation)

    if y_prob is not None:
        try:
            summary["auroc_ovr"] = float(roc_auc_score(y_true, y_prob, multi_class="ovr"))
        except Exception:
            summary["auroc_ovr"] = None
        summary["ece"] = expected_calibration_error(y_true, y_prob, n_bins=10)

    return summary, report


def save_split_outputs(
    split_name,
    y_true,
    y_pred,
    labels,
    label_names,
    out_dir,
    y_prob=None,
    df_meta=None,
):
    summary, report = compute_metrics(y_true, y_pred, labels, label_names, y_prob)

    # Save metrics
    metrics = {
        "split": split_name,
        "summary": summary,
        "report": report,
    }
    with open(out_dir / f"metrics_{split_name}.json", "w") as f:
        json.dump(metrics, f, indent=2)

    # Save per-class report
    per_class = {k: v for k, v in report.items() if k in label_names}
    pd.DataFrame(per_class).T.to_csv(out_dir / f"per_class_{split_name}.csv")

    # Save predictions
    pred_df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
    })
    if df_meta is not None:
        pred_df["img_id"] = df_meta["img_id"].values
        pred_df["image_path"] = df_meta["image_path"].values
    if y_prob is not None:
        for i, name in enumerate(label_names):
            pred_df[f"prob_{name}"] = y_prob[:, i]
    pred_df.to_csv(out_dir / f"pred_{split_name}.csv", index=False)

    return summary, report


In [8]:
labels = sorted(id_to_name.keys())
label_names = [id_to_name[i] for i in labels]

best_val_f1 = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, y_train, pred_train, prob_train = run_epoch(train_dl, model, optimizer)
    val_loss, y_val, pred_val, prob_val = run_epoch(val_dl, model, optimizer=None)

    val_summary, _ = compute_metrics(y_val, pred_val, labels, label_names, prob_val)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_f1": val_summary["macro_f1"],
    })

    if val_summary["macro_f1"] > best_val_f1:
        best_val_f1 = val_summary["macro_f1"]
        torch.save(model.state_dict(), OUT_DIR / "best_resnet50.pt")

    print(f"Epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_macro_f1={val_summary['macro_f1']:.4f}")

# Load best model
model.load_state_dict(torch.load(OUT_DIR / "best_resnet50.pt", map_location=DEVICE))

# Final eval
_, y_train, pred_train, prob_train = run_epoch(train_dl, model, optimizer=None)
_, y_val, pred_val, prob_val = run_epoch(val_dl, model, optimizer=None)
_, y_test, pred_test, prob_test = run_epoch(test_dl, model, optimizer=None)

train_summary, _ = save_split_outputs("train", y_train, pred_train, labels, label_names, OUT_DIR, prob_train, train_df)
val_summary, _ = save_split_outputs("val", y_val, pred_val, labels, label_names, OUT_DIR, prob_val, val_df)
test_summary, _ = save_split_outputs("test", y_test, pred_test, labels, label_names, OUT_DIR, prob_test, test_df)

print("Train summary:")
print(json.dumps(train_summary, indent=2))
print("Val summary:")
print(json.dumps(val_summary, indent=2))
print("Test summary:")
print(json.dumps(test_summary, indent=2))

# Confusion matrices
val_cm = confusion_matrix(y_val, pred_val, labels=labels)
test_cm = confusion_matrix(y_test, pred_test, labels=labels)
np.save(OUT_DIR / "confusion_val.npy", val_cm)
np.save(OUT_DIR / "confusion_test.npy", test_cm)

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

for split_name, cm in [("val", val_cm), ("test", test_cm)]:
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(include_values=False, cmap="Blues", ax=ax, xticks_rotation=90)
    plt.title(f"{split_name.upper()} Confusion Matrix (ResNet50 finetune)")
    plt.tight_layout()
    fig_path = OUT_DIR / f"confusion_{split_name}.png"
    plt.savefig(fig_path, dpi=200)
    plt.close(fig)

# Save run meta
run_meta = {
    "model": "resnet50_finetune",
    "seed": SEED,
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "best_val_macro_f1": best_val_f1,
    "split_hash": (DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "split_hash.txt").read_text().strip()
        if (DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "split_hash.txt").exists() else None,
}
with open(OUT_DIR / "run_meta.json", "w") as f:
    json.dump(run_meta, f, indent=2)

pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)

print("Saved outputs to", OUT_DIR)


Epoch 1: train_loss=0.8619 val_loss=0.6369 val_macro_f1=0.6413
Epoch 2: train_loss=0.7010 val_loss=0.5506 val_macro_f1=0.6604
Epoch 3: train_loss=0.6355 val_loss=0.7009 val_macro_f1=0.6635
Epoch 4: train_loss=0.5840 val_loss=0.6489 val_macro_f1=0.6423
Epoch 5: train_loss=0.5586 val_loss=0.5036 val_macro_f1=0.7455
Epoch 6: train_loss=0.5133 val_loss=0.7009 val_macro_f1=0.6618
Epoch 7: train_loss=0.4703 val_loss=0.6257 val_macro_f1=0.7127
Epoch 8: train_loss=0.4391 val_loss=0.6113 val_macro_f1=0.6698
Epoch 9: train_loss=0.3724 val_loss=0.6241 val_macro_f1=0.7106
Epoch 10: train_loss=0.3618 val_loss=0.6132 val_macro_f1=0.6962
Epoch 11: train_loss=0.3068 val_loss=0.8962 val_macro_f1=0.6407
Epoch 12: train_loss=0.2769 val_loss=0.6966 val_macro_f1=0.6842
Epoch 13: train_loss=0.2314 val_loss=0.6579 val_macro_f1=0.6908
Epoch 14: train_loss=0.2256 val_loss=0.8187 val_macro_f1=0.6427
Epoch 15: train_loss=0.2058 val_loss=0.7590 val_macro_f1=0.7230


/tmp/ipykernel_681731/3544229491.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(OUT_DIR / "best_resnet50.pt", map_location=DEVICE))


Train summary:
{
  "accuracy": 0.8523474449186758,
  "balanced_accuracy": 0.8453099185111456,
  "macro_f1": 0.8362834141690088,
  "weighted_f1": 0.8535872685066581,
  "qwk": 0.9118422668121703,
  "mae": 0.15065174760641364,
  "rmse": 0.3966639056033913,
  "spearman": 0.8679770015687516,
  "auroc_ovr": 0.971238020448677,
  "ece": 0.036785624196804666
}
Val summary:
{
  "accuracy": 0.7904451682953312,
  "balanced_accuracy": 0.740286469244661,
  "macro_f1": 0.7455287106416699,
  "weighted_f1": 0.7915418336464498,
  "qwk": 0.8760429127217427,
  "mae": 0.21715526601520088,
  "rmse": 0.4820333335323035,
  "spearman": 0.8326233946504868,
  "auroc_ovr": 0.9428983549373929,
  "ece": 0.02521388150193403
}
Test summary:
{
  "accuracy": 0.7526690391459074,
  "balanced_accuracy": 0.6857989341620989,
  "macro_f1": 0.6799770705485829,
  "weighted_f1": 0.755411525012814,
  "qwk": 0.8428081870344839,
  "mae": 0.25326215895610915,
  "rmse": 0.5149024715032375,
  "spearman": 0.7906315985078699,
  "auroc_